In [ ]:
"""
07_flufftail_velorama_grn.py
The integration: stage-specific DIRECTIONAL (Granger) GRNs across the transition,
and the headline test — do Flufftail's GRNBoost2 co-variation hubs survive as
directional Velorama regulators?

Reads : liver_endstage.h5ad (step 04), Flufftail hand-off (step 03)
Writes: 99_results/flufftail_velorama/*  (global + per-stage GRNs, hub-influence table)

Same stable regime as steps 04/05 (STABLE_TRAIN_CONFIG), so the global GRN, the
stage GRNs and the definitive GRN are all comparable.
"""
import os, glob, time
from gribben_config import *          # paths, gene sets, hubs, LAG, CAP_REGULATORS, STABLE_TRAIN_CONFIG
import numpy as np, pandas as pd, scipy.sparse as sp
import scanpy as sc, anndata as ad, torch, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from joblib import Parallel, delayed
from velorama import train_model
from velorama.utils import construct_dag, calculate_diffusion_lags
for v in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ[v] = str(N_CPUS)

OUTPUT_DIR   = FLUFF_VELO_DIR
DATASET_NAME = DATASET_CURATED
TRAIN_CONFIG = {**STABLE_TRAIN_CONFIG, "seed": 43}
STAGES = ["hepatocyte", "transition", "cholangiocyte"]

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100


In [ ]:
# === STEP 1: load curated AnnData + Flufftail hand-off ===
adata = ad.read_h5ad(CURATED_H5AD)
if "velorama_reg_names" in adata.uns:
    adata.var["is_reg"]    = adata.var_names.isin(adata.uns["velorama_reg_names"])
    adata.var["is_target"] = adata.var_names.isin(adata.uns["velorama_target_names"])
def _rl(p): return [l.strip() for l in open(p)] if os.path.exists(p) else None
fluff = {"states": pd.read_csv(FLUFF_STATES) if os.path.exists(FLUFF_STATES) else None,
         "pseudotime": pd.read_csv(FLUFF_PSEUDOTIME) if os.path.exists(FLUFF_PSEUDOTIME) else None,
         "de_genes": _rl(FLUFF_DE_GENES), "hubs": _rl(FLUFF_HUBS)}

In [ ]:
# === STEP 2: orient DPT so hepatocytes are the root (pt~0), check Monocle3 concordance ===
hep = adata.obs["cell.annotation"].eq("Hepatocytes")
chol = adata.obs["cell.annotation"].eq("Cholangiocytes")
if adata.obs.loc[hep, "dpt_pseudotime"].mean() > adata.obs.loc[chol, "dpt_pseudotime"].mean():
    adata.obs["dpt_pseudotime"] = adata.obs["dpt_pseudotime"].max() - adata.obs["dpt_pseudotime"]
if fluff["pseudotime"] is not None:                          # cross-method concordance (step 06 axis)
    pt = fluff["pseudotime"].set_index("cell")["pseudotime"].reindex(adata.obs_names)
    ok = pt.notna() & adata.obs["dpt_pseudotime"].notna()
    print(f"Spearman(DPT, Monocle3) = {spearmanr(adata.obs.dpt_pseudotime[ok], pt[ok])[0]:.3f}")

In [ ]:
# === STEP 3: assign three trajectory stages (Flufftail states, else pt terciles) ===
adata.obs["stage"] = "unassigned"
if fluff["states"] is not None:
    s = fluff["states"].set_index("cell")["state"].reindex(adata.obs_names)
    a_first = adata.obs.loc[s.eq("State A").values, "dpt_pseudotime"].mean() < \
              adata.obs.loc[s.eq("State B").values, "dpt_pseudotime"].mean()
    hep_lab, chol_lab = ("State A", "State B") if a_first else ("State B", "State A")
    adata.obs.loc[s.eq(hep_lab).values, "stage"]     = "hepatocyte"
    adata.obs.loc[s.eq("Transition").values, "stage"] = "transition"
    adata.obs.loc[s.eq(chol_lab).values, "stage"]    = "cholangiocyte"
else:
    q = np.quantile(adata.obs["dpt_pseudotime"], [1/3, 2/3])
    adata.obs["stage"] = pd.cut(adata.obs["dpt_pseudotime"], [-np.inf, *q, np.inf], labels=STAGES)

In [ ]:
# --- VIZ: stage assignment on the pseudotime axis ---
fig, ax = plt.subplots(1, 2, figsize=(11,4))
stage_colors = {'hepatocyte':'#e88035','transition':'#9aa0a6','cholangiocyte':'#4C9BE8','unassigned':'lightgrey'}
for st, col in stage_colors.items():
    m = adata.obs.stage.eq(st)
    ax[0].hist(adata.obs.loc[m,'dpt_pseudotime'], bins=40, alpha=.7, label=st, color=col)
ax[0].set_xlabel('pseudotime'); ax[0].legend(fontsize=8); ax[0].set_title('Stage assignment')
adata.obs.stage.value_counts().plot(kind='bar', ax=ax[1],
    color=[stage_colors.get(s,'grey') for s in adata.obs.stage.value_counts().index])
ax[1].set_title('Cells per stage'); plt.tight_layout(); plt.show()


In [ ]:
# === STEP 4: curated regulators + targets (stable environment) ===
# Regulators = lineage TFs + TF hubs, filled by variance to the cap. Non-TF hubs
# (SERPINE1, FGF13) stay as targets — they produced artifact out-edges as regulators.
_X = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
gene_var  = pd.Series(np.nan_to_num(_X.var(axis=0)), index=adata.var_names)
hubs_tf   = [h for h in FLUFFTAIL_HUBS_TF if h in adata.var_names]
must_keep = (LINEAGE_TFS | set(hubs_tf)) & set(adata.var_names)
reg_pool  = [g for g in adata.var_names[adata.var.is_reg.values] if g not in must_keep]
curated   = sorted(must_keep | set(gene_var[reg_pool].sort_values(ascending=False)
                                   .index[:max(0, CAP_REGULATORS - len(must_keep))]))
adata.var["is_reg"]    = adata.var_names.isin(curated)
adata.var["is_target"] = adata.var.is_target | adata.var_names.isin(FLUFFTAIL_HUBS_NONTF)
adata.var.loc[adata.var.is_reg, "is_target"] = False
if fluff["de_genes"]:                                        # focus targets on Flufftail DE genes
    keep = adata.var.is_target & adata.var_names.isin(set(fluff["de_genes"]) | set(FLUFFTAIL_HUBS_NONTF))
    if keep.sum() >= 50: adata.var["is_target"] = keep
reg_idx    = np.where(adata.var.is_reg.values)[0]
target_idx = np.where(adata.var.is_target.values)[0]
print(f"Regulators: {len(reg_idx)} | targets: {len(target_idx)}")

In [ ]:
# --- VIZ: curated regulator/target composition for the integration ---
plt.figure(figsize=(5,3.5))
plt.bar(['regulators','targets'], [len(reg_idx), len(target_idx)], color=['#3576b0','#e88035'])
plt.title('Curated GRN gene universe (step 07)'); plt.tight_layout(); plt.show()


In [ ]:
# === STEP 5: the causal-GRN engine (restricts predictors to regs+targets) ===
def run_grn(a_full, cell_mask, tgt_idx, tag, config=None):
    """Train a Granger GRN on the masked cells; returns regulators x targets scores + lags."""
    cfg = {**TRAIN_CONFIG, **(config or {})}
    reg_names, tgt_names = list(a_full.var_names[reg_idx]), list(a_full.var_names[tgt_idx])
    universe = sorted(set(reg_names) | set(tgt_names))         # keystone conditioning fix
    a = a_full[cell_mask][:, universe].copy()
    pos = {g: i for i, g in enumerate(a.var_names)}; reg_pos = [pos[r] for r in reg_names]
    sc.pp.scale(a)
    npc = min(20, a.n_vars - 1, a.n_obs - 1)
    sc.pp.pca(a, n_comps=npc); sc.pp.neighbors(a, n_neighbors=min(15, a.n_obs-1), n_pcs=npc)
    sc.tl.diffmap(a, n_comps=min(10, npc)); a.uns["iroot"] = int(np.argmin(a.obs.dpt_pseudotime.values))
    Xn = np.ascontiguousarray(np.nan_to_num(
        a.X.toarray() if sp.issparse(a.X) else np.asarray(a.X)).astype(np.float32))
    A  = construct_dag(a, dynamics="pseudotime", ptloc=None, proba=False)
    AX = calculate_diffusion_lags(A, torch.tensor(Xn), LAG)
    AXn = AX.numpy() if isinstance(AX, torch.Tensor) else np.asarray(AX, dtype=np.float32)

    pt_dir = os.path.join(OUTPUT_DIR, tag); os.makedirs(pt_dir, exist_ok=True)
    def _load(g):
        h = [p for p in glob.glob(os.path.join(pt_dir, f"{g}.*lag{LAG}.pseudotime.pt"))
             if not p.endswith(".ignore_lag.pt")]
        return torch.load(h[0], map_location="cpu").detach().numpy()[0] if h else None
    todo = [g for g in tgt_names if _load(g) is None]
    Parallel(n_jobs=max(1, min(3, N_CPUS)), backend="loky")(
        delayed(lambda p, g: train_model(
            {"AX": torch.tensor(AXn), "AY": torch.tensor(AXn),
             "Y": torch.tensor(Xn[:, [p]], dtype=torch.float32), "name": g,
             "results_dir": OUTPUT_DIR, "dir_name": tag, **cfg}))(pos[g], g) for g in todo)
    grn = pd.DataFrame(0.0, index=reg_names, columns=tgt_names)
    lagm = pd.DataFrame(0.0, index=reg_names, columns=tgt_names)
    for g in tgt_names:
        GC = _load(g)
        if GC is None: continue
        grn[g] = GC.max(axis=1)[reg_pos]; lagm[g] = GC.argmax(axis=1)[reg_pos]
    print(f"[{tag}] {a.n_obs} cells | nonzero={ (grn.values!=0).mean():.2f}")
    return grn, lagm

In [ ]:
# === STEP 6: global directional GRN (baseline) ===
grn_g, lag_g = run_grn(adata, np.ones(adata.n_obs, bool), target_idx, f"{DATASET_NAME}_global")
grn_g.to_csv(os.path.join(OUTPUT_DIR, "grn_global.csv"))

In [ ]:
# --- VIZ: global GRN heatmap (top regulators x top targets) ---
import seaborn as sns
top_r = grn_g.sum(axis=1).nlargest(20).index
top_t = grn_g.loc[top_r].sum(axis=0).nlargest(30).index
plt.figure(figsize=(10,6))
sns.heatmap(grn_g.loc[top_r, top_t], cmap='viridis', cbar_kws={'label':'GRN score'})
plt.title('Global directional GRN (top regs x top targets)'); plt.tight_layout(); plt.show()


In [ ]:
# === STEP 7: permutation null -> significance threshold (pseudotime shuffled) ===
rng = np.random.default_rng(99)
a_null = adata.copy(); a_null.obs["dpt_pseudotime"] = rng.permutation(a_null.obs.dpt_pseudotime.values)
sel = np.sort(rng.choice(a_null.n_obs, min(3000, a_null.n_obs), replace=False))
nmask = np.zeros(adata.n_obs, bool); nmask[sel] = True
ntgt = rng.choice(target_idx, min(30, len(target_idx)), replace=False)
grn_null, _ = run_grn(a_null, nmask, np.array(ntgt), f"{DATASET_NAME}_null", {"seed": 99})
ns = grn_null.values.flatten(); ns = ns[ns > 0]
THRESH = float(np.percentile(ns, 99))
print(f"Significance threshold (null p99) = {THRESH:.4f}")

In [ ]:
# --- VIZ: null vs real distribution, and the threshold it sets ---
plt.figure(figsize=(7,4))
plt.hist(ns, bins=80, alpha=.6, density=True, label='null', color='grey')
plt.hist(grn_g.values.flatten(), bins=80, alpha=.6, density=True, label='global GRN', color='#3576b0')
plt.axvline(THRESH, color='crimson', linestyle='--', label=f'p99 = {THRESH:.3f}')
plt.legend(); plt.title('Permutation null vs. global GRN scores'); plt.tight_layout(); plt.show()


In [ ]:
# === STEP 8: stage-specific directional GRNs ===
stage_long = {}
for st in STAGES:
    mask = adata.obs["stage"].eq(st).values
    if mask.sum() < 100:
        print(f"skip {st}: {mask.sum()} cells"); continue
    g, l = run_grn(adata, mask, target_idx, f"{DATASET_NAME}_{st}")
    g.to_csv(os.path.join(OUTPUT_DIR, f"grn_{st}.csv"))
    d = g.stack().reset_index(); d.columns = ["TF", "target", "grn_score"]
    d = d.merge(l.stack().reset_index().set_axis(["TF", "target", "lag_score"], axis=1),
                on=["TF", "target"], how="left")
    d = d[d.TF != d.target]; d["stage"] = st; d["significant"] = d.grn_score > THRESH
    stage_long[st] = d
    print(f"{st}: {int(d.significant.sum())} significant edges")

In [ ]:
# --- VIZ: significant-edge counts per stage ---
plt.figure(figsize=(5,3.5))
counts = {st: int(d.significant.sum()) for st, d in stage_long.items()}
plt.bar(counts.keys(), counts.values(), color=['#e88035','#9aa0a6','#4C9BE8'][:len(counts)])
plt.title('Significant directional edges per stage'); plt.tight_layout(); plt.show()


In [ ]:
# === STEP 9: do Flufftail hubs survive as directional regulators, and how do they rewire? ===
hubs = [h for h in (fluff["hubs"] or FLUFFTAIL_HUBS) if h in adata.var_names[reg_idx]]
watch = hubs + sorted(LINEAGE_TFS & set(adata.var_names[reg_idx]))
rows = []
for st, d in stage_long.items():
    sig = d[d.significant]
    for tf in watch:
        sub = sig[sig.TF == tf]
        rows.append(dict(TF=tf, stage=st, out_degree=len(sub),
                         total_influence=sub.grn_score.sum(),
                         mean_lag=sub.lag_score.mean() if len(sub) else np.nan,
                         is_flufftail_hub=tf in hubs))
infl = pd.DataFrame(rows)
infl.to_csv(os.path.join(OUTPUT_DIR, "hub_influence_by_stage.csv"), index=False)
if not infl.empty:
    piv = infl.pivot_table(index="TF", columns="stage", values="total_influence", fill_value=0) \
              .reindex(columns=[s for s in STAGES if s in infl.stage.unique()])
    print("\nDirectional influence by stage (Flufftail hubs survive if rows are non-zero):")
    print(piv.round(3).to_string())
    ax = piv.plot(kind="bar", figsize=(9, 5)); ax.set_ylabel("Sum significant Granger score")
    ax.set_title("Regulator rewiring across stages (Velorama-validated)")
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "hub_influence_by_stage.png"),
                                    dpi=150, bbox_inches="tight"); plt.close()

## NEXT STEPS / GAPS
## - PER_STAGE_NULL is off: all stages share the global threshold. Each stage has
##   fewer cells (looser null); compute a per-stage null for a strict per-stage FPR.
## - Set USE_MONOCLE3 logic (see step 07 notebook) to drive the DAG with Flufftail's
##   Monocle3 pseudotime instead of DPT, and compare the resulting GRNs.
## - Directionality/lag is only meaningful if step 06 confirmed a shared trajectory
##   and step 05 marked the regime STABLE — check both before interpreting edges.
## - Compare Flufftail's undirected GRNBoost2 hubs vs these directional regulators
##   explicitly (overlap + which edges gained a direction/lag) for the headline result.